In [13]:
import pandas as pd
import numpy as np
# 1. Cargar los datos (usamos sep='\t' porque los datos están tabulados)
df = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/dataset.csv', sep = '\t')

# 2. Pivotar los datos de formato 'melt' a matriz de características (N x P)
# Las filas serán pacientes y las columnas serán las bacterias
df_wide = df.pivot(
    index=['Paciente', 'Diagnosticos'], 
    columns='Bacteria_Limpia', 
    values='AbundanciaTaxonomica'
).reset_index()

# Llenar posibles valores nulos con 0 (en caso de que falten bacterias en algunos pacientes)
df_wide = df_wide.fillna(0)


In [15]:
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC

# Descartamos las columnas identificadoras para quedarnos solo con las abundancias
X = df_wide.drop(columns=['Paciente', 'Diagnosticos'])
y = df_wide['Diagnosticos'] # 1 = TEA, 0 = HC

print(f"Matriz X lista con dimensiones: {X.shape} (Pacientes x Bacterias)")
print(f"Distribución de etiquetas (y):\n{y.value_counts()}\n")

# Configurar el SVM Lineal y la Validación Cruzada (5-Fold Estratificado)
svm_lineal = SVC(kernel='linear', class_weight='balanced')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#  Ejecutar los exámenes (Cross-Validation)
notas_cv = cross_val_score(svm_lineal, X, y, cv=skf, scoring='accuracy')

print("=== RESULTADOS DE VALIDACIÓN CRUZADA ===")
print(f"Precisiones por iteración: {np.round(notas_cv, 3)}")
print(f"Precisión Promedio Esperada: {np.mean(notas_cv):.3f} +/- {np.std(notas_cv):.3f}\n")

#Entrenar el Modelo Definitivo con el 100% de los datos
modelo_definitivo = svm_lineal.fit(X, y)

pesos = modelo_definitivo.coef_[0]
nombres_bacterias = X.columns

df_importancias = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Peso': pesos
}).sort_values(by='Peso', ascending=False)

# Mostrar las 5 bacterias que más predicen TEA (pesos positivos altos)
print("=== TOP 5 BIOMARCADORES PARA TEA ===")
print(df_importancias.head(5))

# Mostrar las 5 bacterias que más predicen HC (pesos negativos altos)
print("\n=== TOP 5 BIOMARCADORES PARA HC ===")
print(df_importancias.tail(5))

Matriz X lista con dimensiones: (121, 153) (Pacientes x Bacterias)
Distribución de etiquetas (y):
Diagnosticos
1    71
0    50
Name: count, dtype: int64

=== RESULTADOS DE VALIDACIÓN CRUZADA ===
Precisiones por iteración: [0.72  0.792 0.792 0.75  0.792]
Precisión Promedio Esperada: 0.769 +/- 0.029

=== TOP 5 BIOMARCADORES PARA TEA ===
                             Bacteria      Peso
56                     Frisingicoccus  0.242774
26                Christensenellaceae  0.201051
148  [Eubacterium]_xylanophilum_group  0.197316
93                       Ochrobactrum  0.164453
7                           Alistipes  0.157219

=== TOP 5 BIOMARCADORES PARA HC ===
                         Bacteria      Peso
129                    Tyzzerella -0.169111
51                   Faecalitalea -0.173492
3                   Adlercreutzia -0.185376
142   [Eubacterium]_eligens_group -0.190472
74   Lachnospiraceae_ND3007_group -0.203966
